# PySpark: detailed scenarios

Distributed DataFrames in local mode. **Requires the pyspark package and a Java 11 or 17 runtime.** If pyspark is not installed, run `poetry add pyspark` and install a JDK. See **pyspark_cheatsheet.md**.

In [ ]:
from pyspark.sql import SparkSession, functions as F, Window

spark = SparkSession.builder.master("local[*]").appName("explore").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

## Build a DataFrame

In [ ]:
import random
random.seed(4)
rows = []
for i in range(300):
    t = random.choice(["ACME", "GLOBEX", "INITECH"])
    rows.append((t, i, round(random.gauss(0.0004, 0.012), 5)))

df = spark.createDataFrame(rows, ["ticker", "day", "ret"])
df.show(3)

## Select, filter, withColumn

In [ ]:
df.filter(F.col("ret") > 0).select("ticker", "ret").show(3)
df.withColumn("up", F.col("ret") > 0).groupBy("up").count().show()

## Group and aggregate

In [ ]:
df.groupBy("ticker").agg(
    F.mean("ret").alias("mean"),
    F.stddev("ret").alias("vol"),
    F.count("*").alias("n"),
).show()

## Window functions: lag and a running average

In [ ]:
w = Window.partitionBy("ticker").orderBy("day")
(df.withColumn("prev", F.lag("ret").over(w))
   .withColumn("run_avg", F.avg("ret").over(w.rowsBetween(-9, 0)))
   .filter(F.col("ticker") == "ACME")
   .orderBy("day")
   .show(5))

## SQL

In [ ]:
df.createOrReplaceTempView("r")
spark.sql("select ticker, round(avg(ret), 5) as mean from r group by ticker order by mean desc").show()

## Vectorised pandas_udf

In [ ]:
import pandas as pd
from pyspark.sql.functions import pandas_udf

@pandas_udf("double")
def demean(s: pd.Series) -> pd.Series:
    return s - s.mean()

df.withColumn("demeaned", demean("ret")).show(3)

## Collect a small result and stop Spark

In [ ]:
pdf = df.groupBy("ticker").agg(F.mean("ret").alias("mean")).toPandas()
print(pdf)
spark.stop()

## Recap

A local SparkSession, build/select/filter, groupBy, window functions, SQL, a vectorised pandas_udf, and toPandas. The same code scales to a cluster by changing the master.